In [1]:
print("Hi")

Hi


In [2]:
from dotenv import load_dotenv
import os
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
# os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
# os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
print("Environment variables loaded successfully.")

Environment variables loaded successfully.


In [3]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent

llm = ChatOpenAI(model="gpt-4o-mini")


# =========================
# STEP 1: DEFINE TOOLS
# =========================

# --- Math tools ---
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


# --- Info tool (mock RAG) ---
@tool
def rag_search(query: str) -> str:
    """Answer questions about LangSmith"""
    return "LangSmith helps monitor, debug, and deploy LLM applications."


# =========================
# STEP 2: CREATE SPECIALIZED AGENTS
# =========================

# Math Agent
math_agent = create_agent(
    model=llm,
    tools=[add, multiply],
    system_prompt="You are a math expert. Solve only math problems."
)

# Info Agent
info_agent = create_agent(
    model=llm,
    tools=[rag_search],
    system_prompt="You are a knowledge assistant. Answer general questions."
)


# =========================
# STEP 3: ROUTER (LLM decides)
# =========================

def router_agent(question: str):
    routing_prompt = f"""
    Decide which agent should handle this question:
    - math → for calculations
    - info → for general knowledge

    Question: {question}

    Answer only: math or info
    """

    decision = llm.invoke(routing_prompt).content.strip().lower()

    print("\n🧠 Router Decision:", decision)

    if "math" in decision:
        agent = math_agent
    else:
        agent = info_agent

    # Call selected agent
    response = agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })

    return response["messages"][-1].content


# =========================
# STEP 4: TEST
# =========================

questions = [
    "What is 5 + 7?",
    "Multiply 6 and 9",
    "What is LangSmith?",
    "Explain LangSmith deployment"
]

for q in questions:
    print("\n=========================")
    print("Question:", q)
    answer = router_agent(q)
    print("✅ Answer:", answer)


Question: What is 5 + 7?

🧠 Router Decision: math
✅ Answer: 5 + 7 equals 12.

Question: Multiply 6 and 9

🧠 Router Decision: math
✅ Answer: The product of 6 and 9 is 54.

Question: What is LangSmith?

🧠 Router Decision: info
✅ Answer: LangSmith is a platform that helps monitor, debug, and deploy applications built on large language models (LLMs).

Question: Explain LangSmith deployment

🧠 Router Decision: info
✅ Answer: LangSmith assists in monitoring, debugging, and deploying applications that utilize large language models (LLMs). This involves providing tools and frameworks for effectively managing the lifecycle of LLM applications, ensuring they run smoothly and efficiently in various environments.
